# Insurgent Primaries, 2018–2026: Money, Outside Spending, and Outcomes

The case universe is every House candidate endorsed by Justice Democrats from 2018 through 2026 (144 endorsement rows from the organization's Wikipedia tables, which carry primary result and vote share), matched to FEC data for the candidate and for their main Democratic-primary opponent — the incumbent if the JD candidate was a challenger, else the best-funded other Democrat in the district. Justice Democrats is used as the universe because it is the one insurgent-left organization with a complete, dated endorsement record across all five cycles; DSA's federal endorsements overlap heavily with it (Ocasio-Cortez, Tlaib, Bush, Bowman, Lee, Casar, Kiros, Valdez, Avila Chevalier, McKinney).

**Questions.** Do JD candidates win when outspent, and how often? Does outside spending against them predict losing? How do the money ratios in the wins (Ocasio-Cortez 2018, Bowman 2020, Bush 2020, Lee 2022, Kiros / McKinney / Avila Chevalier 2026) compare with the losses (Cisneros 2020 and 2022, Turner 2021, Bowman and Bush 2024, Allam and Abughazaleh 2026)?

**Inputs:** `data/raw/insurgent/jd_cases_fec.csv` from `src/data_collection/fec_insurgent_pull.py`. Money is receipts through the last FEC report filed before the primary (year-to-date plus the prior year-end), which is the cleanest pre-primary measure available without itemized-transaction pulls; independent expenditures are Schedule E totals for the cycle by support/oppose.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
pd.set_option('display.width', 240)
d = pd.read_csv(RAW_DIR / 'insurgent' / 'jd_cases_fec.csv')
# 'Advanced' = made a top-two general (CA/WA) or a runoff; runoff outcomes and blank results are set by hand from the Wikipedia table
OVERRIDE = {(2022, 'Jessica Cisneros'): ('Lost', 49.69), (2022, 'Kina Collins'): ('Lost', 45.72), (2022, 'Marie Newman'): ('Lost', 29.23), (2018, 'Lorie Burch'): ('Won', 75.04)}
for (cy, nm), (res, pct) in OVERRIDE.items():
    d.loc[(d.cycle == cy) & (d.candidate == nm), ['prim_result', 'prim_pct']] = [res, f'{pct}%']
# Ideological tier. 'DSA' = a DSA member or DSA-endorsed (national or chapter) per the public record
# (Ballotpedia's DSA endorsement tables 2018-2026; DSA statements; candidates' Wikipedia articles). Everyone else in the
# universe is a Justice Democrats-only progressive (JD's criteria already require Medicare for All and no corporate PAC money).
DSA = {'Alexandria Ocasio-Cortez', 'Rashida Tlaib', 'Cori Bush', 'Jamaal Bowman', 'Ilhan Omar', 'Summer Lee', 'Greg Casar', 'Melat Kiros', 'Donavan McKinney',
       'Chris Rabb', 'Claire Valdez', 'Darializa Avila Chevalier', 'Kaniela Ing', 'Sarah Smith'}
d['tier'] = np.where(d.candidate.isin(DSA), 'DSA', 'JD-only')
# Top-two 'Advanced' for a non-incumbent (CA/WA) is neither a primary win nor a loss; excluded from win rates
d['pending'] = (d.prim_result == 'Advanced') & (~d.jd_incumbent) & d.state.isin(['CA', 'WA'])
d['won'] = d.prim_result.isin(['Won', 'Advanced']).astype(float)
d['pct'] = pd.to_numeric(d.prim_pct.astype(str).str.rstrip('%'), errors='coerce')
d['contested'] = (d.pct < 99) & (~d.pending)
d['type'] = np.select([d.jd_incumbent, d.opp_is_incumbent], ['JD incumbent', 'challenger vs incumbent'], 'open / other')
for c in ['pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support', 'ie_oppose', 'opp_ie_support', 'opp_ie_oppose', 'cycle_receipts', 'opp_cycle_receipts',
          'ie_support_pre', 'ie_oppose_pre', 'opp_ie_support_pre', 'opp_ie_oppose_pre']:
    d[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) / 1e6
# Money sides use independent expenditures made on or before the primary date (itemized Schedule E), not cycle totals
d['jd_side'] = d.pre_primary_receipts + d.ie_support_pre + d.opp_ie_oppose_pre
d['opp_side'] = d.opp_pre_primary_receipts + d.opp_ie_support_pre + d.ie_oppose_pre
d['money_ratio'] = d.opp_side / d.jd_side.replace(0, np.nan)
d['outspent'] = d.opp_side > d.jd_side
d['small_share'] = pd.to_numeric(d.pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.pre_primary_receipts, errors='coerce') * 1e6)
print(len(d), 'cases;', d.contested.sum(), 'contested;', d.pending.sum(), 'pending top-two')
print(d.groupby(['cycle', 'type']).size().unstack(fill_value=0))
print(d.groupby(['tier', 'type']).size().unstack(fill_value=0))

133 cases; 106 contested; 5 pending top-two
type   JD incumbent  challenger vs incumbent  open / other
cycle                                                     
2018              3                        9            54
2020              7                        6             1
2021              0                        0             1
2022             10                        3             3
2024             12                        0             0
2026              9                        7             8
type     JD incumbent  challenger vs incumbent  open / other
tier                                                        
DSA                20                        9             7
JD-only            21                       16            60


## 1. Win rates by case type, cycle, and money position

In [2]:
c = d[d.contested].copy()
print('Contested primaries only (JD candidate < 99%):')
print(c.groupby('type').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean(), 'median_money_ratio': s.money_ratio.median()})).round(2))
print()
print('By ideological tier (DSA member/endorsed vs JD-only):')
print(c.groupby(['tier', 'type']).apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean(), 'median_money_ratio': s.money_ratio.median(), 'mean_pct': s.pct.mean()})).round(2))
print()
print(c.groupby('cycle').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean()})).round(2))
print()
print('Win rate by whether the JD side was outspent (contested):')
print(c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean(), 'mean_pct': s.pct.mean()})).round(2))
c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean()})).to_csv(TABLES_DIR / 'insurgent_winrate_by_money.csv')

Contested primaries only (JD candidate < 99%):
                            N  wins  win_rate  outspent_share  median_money_ratio
type                                                                             
JD incumbent             28.0  25.0      0.89            0.11                0.01
challenger vs incumbent  21.0   8.0      0.38            0.76                2.45
open / other             57.0  19.0      0.33            0.70                1.62

By ideological tier (DSA member/endorsed vs JD-only):
                                    N  wins  win_rate  outspent_share  median_money_ratio  mean_pct
tier    type                                                                                       
DSA     JD incumbent             17.0  15.0      0.88            0.12                0.03     66.20
        challenger vs incumbent   8.0   6.0      0.75            1.00                2.32     48.59
        open / other              7.0   5.0      0.71            0.43                0.7

## 2. The challenger-vs-incumbent cases, one by one

In [3]:
ch = c[c.type == 'challenger vs incumbent'].sort_values(['cycle', 'pct'], ascending=[True, False])
cols = ['cycle', 'candidate', 'tier', 'state', 'district', 'opp_name', 'prim_result', 'pct', 'pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support_pre', 'ie_oppose_pre', 'opp_ie_support_pre', 'opp_ie_oppose_pre', 'money_ratio', 'small_share', 'ie_top_opposers']
print(ch[cols].round(2).to_string(index=False))
ch[cols].to_csv(TABLES_DIR / 'insurgent_challengers_vs_incumbents.csv', index=False)

 cycle                 candidate    tier state  district                         opp_name prim_result   pct  pre_primary_receipts  opp_pre_primary_receipts  ie_support_pre  ie_oppose_pre  opp_ie_support_pre  opp_ie_oppose_pre  money_ratio  small_share                                                                                     ie_top_opposers
  2018           Ayanna Pressley JD-only    MA         7               CAPUANO, MICHAEL E         Won 58.60                  0.90                      2.08            0.06           0.00                0.28               0.00         2.45         0.22                                                                                                 NaN
  2018  Alexandria Ocasio-Cortez     DSA    NY        14                  CROWLEY, JOSEPH         Won 56.74                  0.37                      5.54            0.00           0.00                0.00               0.01        14.40         0.69                                             

## 3. Outside spending against insurgents: who, how much, and did it work?

In [4]:
d['ie_against_any'] = d.ie_oppose_pre > 0.25
print('Cases with > $250k in independent expenditures against the JD candidate BEFORE the primary:')
big = d[d.ie_against_any].sort_values('ie_oppose_pre', ascending=False)
print(big[['cycle', 'candidate', 'state', 'district', 'type', 'tier', 'prim_result', 'pct', 'ie_oppose_pre', 'ie_oppose', 'ie_support_pre', 'jd_side', 'opp_side', 'ie_top_opposers']].round(2).to_string(index=False))
print('\nWin rate: with > $250k against %.2f (N=%d); without %.2f (N=%d) [contested only]' % (
    c[c.ie_oppose_pre > 0.25].won.mean(), (c.ie_oppose_pre > 0.25).sum(), c[c.ie_oppose_pre <= 0.25].won.mean(), (c.ie_oppose_pre <= 0.25).sum()))
# who spends against: aggregate top opposers text
import re
from collections import Counter
cnt = Counter()
for s in d.ie_top_opposers.dropna():
    for part in s.split(';'):
        m = re.match(r'\s*(.+?)\s+([\d.]+)M', part)
        if m: cnt[m.group(1).strip()] += float(m.group(2))
print('\nLargest opposing committees across all cases ($M, top-3 per case summed):')
for k, v in cnt.most_common(12): print(f'  {k[:50]:50s} {v:6.2f}')

Cases with > $250k in independent expenditures against the JD candidate BEFORE the primary:
 cycle                 candidate state  district                    type    tier prim_result   pct  ie_oppose_pre  ie_oppose  ie_support_pre  jd_side  opp_side                                                                                                          ie_top_opposers
  2024             Jamaal Bowman    NY        16            JD incumbent     DSA        Lost 41.41          12.04      12.04            1.95     8.88     24.84                                                  UNITED DEMOCRACY PROJECT ('UDP') 9.87M; FAIRSHAKE 2.08M; DMFI PAC 0.10M
  2024                 Cori Bush    MO         1            JD incumbent     DSA        Lost 45.64           7.34       7.34            2.38     7.46     17.48                                  UNITED DEMOCRACY PROJECT ('UDP') 5.24M; FAIRSHAKE 1.41M; MAINSTREAM DEMOCRATS PAC 0.60M
  2026                 Cori Bush    MO         1 challenger vs in

## 4. Models: money ratio, outside spending, incumbency, and the vote share

In [5]:
m = c.dropna(subset=['money_ratio']).copy()
m['log_ratio'] = np.log(m.money_ratio.clip(0.05, 50))
m['log_ie_oppose'] = np.log1p(m.ie_oppose_pre)
m['dsa'] = (m.tier == 'DSA').astype(float)
m['vs_inc'] = (m.type == 'challenger vs incumbent').astype(float); m['jd_inc'] = (m.type == 'JD incumbent').astype(float)
m['post2020'] = (m.cycle >= 2022).astype(float)
X = sm.add_constant(m[['log_ratio', 'log_ie_oppose', 'vs_inc', 'jd_inc', 'post2020', 'dsa']])
r = sm.OLS(m.pct, X).fit(cov_type='HC1'); print('Primary vote share (%%) on money and type, contested cases (N=%d):' % len(m)); print(pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(2))
r2 = sm.Logit(m.won, X).fit(disp=0, cov_type='HC1'); print('\nWin (logit):'); print(pd.DataFrame({'coef': r2.params, 'se': r2.bse, 'p': r2.pvalues}).round(2))
pd.DataFrame({'ols_coef': r.params, 'ols_se': r.bse, 'logit_coef': r2.params, 'logit_se': r2.bse}).to_csv(TABLES_DIR / 'insurgent_models.csv')

Primary vote share (%) on money and type, contested cases (N=106):
                coef    se     p
const          35.43  2.44  0.00
log_ratio      -7.87  0.99  0.00
log_ie_oppose  -2.91  2.06  0.16
vs_inc         12.84  2.83  0.00
jd_inc         11.06  4.20  0.01
post2020        0.31  2.77  0.91
dsa             6.47  3.26  0.05

Win (logit):
               coef    se     p
const         -0.59  0.43  0.16
log_ratio     -0.85  0.27  0.00
log_ie_oppose -2.15  0.66  0.00
vs_inc        -0.01  0.74  0.99
jd_inc         1.19  0.75  0.11
post2020       0.26  0.59  0.66
dsa            2.98  1.21  0.01


## 5. Small-dollar share

In [6]:
print('Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:')
d['opp_small_share'] = pd.to_numeric(d.opp_pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.opp_pre_primary_receipts, errors='coerce') * 1e6)
print(d.groupby('type')[['small_share', 'opp_small_share']].median().round(2))
print(d.groupby('cycle')[['small_share', 'opp_small_share']].median().round(2))
print('\nWinners vs losers among contested JD candidates (median):'); print(c.groupby('won')[['small_share', 'pre_primary_receipts', 'money_ratio', 'ie_oppose_pre']].median().round(2))
print('\nBy tier (median, contested):'); print(c.groupby('tier')[['small_share', 'pre_primary_receipts', 'money_ratio', 'ie_oppose_pre', 'pct', 'won']].median().round(2))

Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:
                         small_share  opp_small_share
type                                                 
JD incumbent                    0.28             0.05
challenger vs incumbent         0.39             0.02
open / other                    0.35             0.12
       small_share  opp_small_share
cycle                              
2018          0.35             0.12
2020          0.41             0.01
2021          0.56             0.04
2022          0.32             0.05
2024          0.24             0.07
2026          0.32             0.03

Winners vs losers among contested JD candidates (median):
     small_share  pre_primary_receipts  money_ratio  ie_oppose_pre
won                                                               
0.0         0.35                  0.50         2.73            0.0
1.0         0.34                  2.53         0.20            0.0

By tier (median, contested):


## 6. Media coverage of the challenger cases

GDELT coverage volume and tone for each challenger and the incumbent they ran against, over the 90 days ending on the primary date. The window stops at the primary deliberately: coverage after a result is largely a consequence of it (AOC's volume multiplied roughly a hundredfold the week she won), so it cannot go into a variable about the result.

Coverage turned out to be far deeper than expected for House primaries — 41 of 42 candidate-series have at least 20 days with coverage, the exception being Sameena Mustafa (IL-5 2018) at three. Median covered days are 45 for challengers and 64 for incumbents.

Two limits carry through everything below. Queries are name phrases with no district context, so common names pick up unrelated coverage — a particular risk for the incumbents Danny Davis, Mike Quigley and Wesley Bell, and for the challenger Marie Newman, whose 2018 volume exceeds Lipinski's by eight to one in a race she lost by two points. And GDELT tone is a lexicon score on article text: as Part 10 documents at length, it measures word valence, not framing, and cannot tell an attack from a report of one.

In [7]:
from scipy import stats
GF = RAW_DIR / 'gdelt_insurgent' / 'gdelt_insurgent_daily.csv'
if not GF.exists():
    print('insurgent GDELT file not on disk; run src/data_collection/gdelt_insurgent_pull.py')
else:
    gi = pd.read_csv(GF)
    def wtone(x):
        ok = (x.volume > 0) & x.tone.notna()
        return np.average(x.tone[ok], weights=x.volume[ok]) if ok.any() else np.nan
    si = gi.groupby(['case_id','role','name','tier','won']).apply(lambda x: pd.Series({
        'covered': int((x.volume > 0).sum()), 'vol': x.volume.mean()*1000, 'tone': wtone(x)})).reset_index()
    print(f'candidate-series: {len(si)}; with >=10 covered days: {(si.covered>=10).sum()}')
    si = si[si.covered >= 10]
    chs, ins = si[si.role=='challenger'], si[si.role=='incumbent']

    print('\n1. CHALLENGER COVERAGE BY TIER')
    rows=[]
    for tier in ['DSA','JD-only']:
        t = chs[chs.tier==tier]
        rows.append({'tier':tier,'n':len(t),'tone':t.tone.mean(),'tone_sd':t.tone.std(),'vol_mean':t.vol.mean(),'vol_median':t.vol.median()})
    tt = pd.DataFrame(rows); print(tt.round(2).to_string(index=False))
    A,B = chs[chs.tier=='DSA'], chs[chs.tier=='JD-only']
    _,p_t = stats.ttest_ind(A.tone,B.tone,equal_var=False); _,p_v = stats.ttest_ind(A.vol,B.vol,equal_var=False)
    print(f"   DSA - JD-only: tone {A.tone.mean()-B.tone.mean():+.2f} (p={p_t:.3f}), volume {A.vol.mean()-B.vol.mean():+.2f} (p={p_v:.3f}) -- null on both")
    tt.to_csv(TABLES_DIR / 'insurgent_media_by_tier.csv', index=False)

    print('\n2. CHALLENGER vs THE INCUMBENT THEY FACED (paired within case)')
    J = chs.merge(ins, on='case_id', suffixes=('_ch','_in'))
    J['tone_gap'] = J.tone_ch - J.tone_in; J['vol_ratio'] = J.vol_ch / J.vol_in
    t_, p_ = stats.ttest_1samp(J.tone_gap.dropna(), 0)
    print(f"   tone gap mean {J.tone_gap.mean():+.2f}, median {J.tone_gap.median():+.2f}; paired t={t_:+.2f} p={p_:.3f}")
    print(f"   {(J.tone_gap<0).sum()} of {len(J)} challengers were covered MORE negatively than their incumbent")
    print(f"   volume ratio challenger/incumbent: median {J.vol_ratio.median():.2f}")
    for tier in ['DSA','JD-only']:
        t = J[J.tier_ch==tier]
        print(f"     {tier:8s} tone gap {t.tone_gap.mean():+.2f}, volume ratio median {t.vol_ratio.median():.2f}")
    J.to_csv(TABLES_DIR / 'insurgent_media_paired.csv', index=False)

    print('\n3. BY OUTCOME')
    for w,lab in [(1,'won'),(0,'lost')]:
        t = chs[chs.won==w]; print(f"   {lab:5s} n={len(t):2d}  tone {t.tone.mean():+.2f}  volume {t.vol.mean():.2f}")
    W,L = chs[chs.won==1], chs[chs.won==0]
    _,p1 = stats.ttest_ind(W.tone,L.tone,equal_var=False); _,p2 = stats.ttest_ind(W.vol,L.vol,equal_var=False)
    print(f"   winners - losers: tone {W.tone.mean()-L.tone.mean():+.2f} (p={p1:.3f}), volume {W.vol.mean()-L.vol.mean():+.2f} (p={p2:.3f})")
    print("\n   The outcome difference is null before any control is applied, so no model of outcome on")
    print("   tone net of money is fitted: with 20 cases and 8 wins it would be fitting noise to a")
    print("   descriptive null, and pre-primary coverage partly reflects expected viability anyway.")

    print('\n4. PER CASE')
    print(J[['case_id','tier_ch','won_ch','name_ch','tone_ch','tone_in','tone_gap','vol_ch','vol_in']]
          .round(2).sort_values('tone_gap').to_string(index=False))

candidate-series: 42; with >=10 covered days: 41

1. CHALLENGER COVERAGE BY TIER
   tier  n  tone  tone_sd  vol_mean  vol_median
    DSA  8 -1.10     0.87      1.69        1.18
JD-only 12 -0.89     0.55      1.95        1.42
   DSA - JD-only: tone -0.21 (p=0.558), volume -0.27 (p=0.660) -- null on both

2. CHALLENGER vs THE INCUMBENT THEY FACED (paired within case)
   tone gap mean +0.70, median +0.42; paired t=+3.71 p=0.001
   5 of 20 challengers were covered MORE negatively than their incumbent
   volume ratio challenger/incumbent: median 0.69
     DSA      tone gap +1.16, volume ratio median 0.64
     JD-only  tone gap +0.39, volume ratio median 0.71

3. BY OUTCOME
   won   n= 8  tone -1.08  volume 2.13
   lost  n=12  tone -0.90  volume 1.66
   winners - losers: tone -0.18 (p=0.625), volume +0.47 (p=0.500)

   The outcome difference is null before any control is applied, so no model of outcome on
   tone net of money is fitted: with 20 cases and 8 wins it would be fitting noise to a

## 7. Framing: what the headlines called them

Tone answers the wrong question. It scores word valence, and Part 10 shows it is nearly blind to framing — an A1 headline leading with a candidate's religion scored negative on the unrelated tokens *war* and *criticism*. What people mean when they say insurgent-left candidates are "framed as radical socialists" is not that the average adjective is negative; it is that a particular description gets reached for.

So this section measures the description. `gdelt_artlist_pull.py` returns one record per article with its headline and domain — 10,516 articles across the twenty-one challengers, 2,145 distinct domains — and `src/analysis/framing.py` classifies each headline.

The design problem is that **calling a DSA member a "democratic socialist" is accurate reporting.** A raw socialist-mention rate would score accurate description and hostile labelling identically, then discover, uselessly, that socialists are called socialists. Labels are therefore split by register: self-description (*democratic socialist*, *progressive*) against loaded variants (*radical*, *far-left*, *Marxist*, *communist*, *extremist*, and bare *socialist* where *democratic* does not precede it), with threat framing (*dangerous*, *takeover*, *purge*) scored separately since fear framing needs no label at all. The summary measure is the **loaded share** — of the headlines that reached for an ideological label, how many reached for the hostile one — which is comparable across candidates whose politics genuinely differ.

In [8]:
import sys
sys.path.insert(0, '../src/analysis')
from framing import score, rates, FRAMES
from scipy import stats

AF = RAW_DIR / 'gdelt_insurgent' / 'gdelt_insurgent_articles.csv'
if not AF.exists():
    print('article-level file not on disk; run src/data_collection/gdelt_artlist_pull.py')
else:
    art = pd.read_csv(AF)
    sc = score(art)
    sc['cycle'] = sc.case_id.str[:4].astype(int)
    print(f'{len(sc)} headlines, {sc.name.nunique()} candidates, {sc.domain.nunique()} domains')

    print('\n1. BY TIER')
    bt = rates(sc, 'tier')
    print(bt[['tier','n_articles','self_description','loaded_label','threat','identity','israel_gaza','loaded_share']].round(3).to_string(index=False))
    D, J = sc[sc.tier=='DSA'], sc[sc.tier=='JD-only']
    for c in ['loaded_label','threat','identity','israel_gaza']:
        _, pv = stats.ttest_ind(D[c].astype(float), J[c].astype(float), equal_var=False)
        print(f'   {c:14s} DSA {D[c].mean():.4f}  JD-only {J[c].mean():.4f}  p={pv:.2g}')
    bt.to_csv(TABLES_DIR / 'insurgent_framing_by_tier.csv', index=False)

    print('\n2. WHAT THE LOADED LABEL ACTUALLY IS')
    td = D.title.fillna('').str.lower()
    HARD = r'\bradical|\bfar[\s-]left|\bmarxis|\bcommunis|\bextremis'
    bare = td.str.contains(r'\bsocialis', regex=True, na=False) & ~td.str.contains(r'democratic[\s-]socialis', regex=True, na=False)
    hard = td.str.contains(HARD, regex=True, na=False)
    jh = J.title.fillna('').str.lower().str.contains(HARD, regex=True, na=False)
    print(f"   of {int(D.loaded_label.sum())} loaded DSA headlines: {int((bare & D.loaded_label).sum())} are bare 'socialist', "
          f"{int((hard & D.loaded_label).sum())} are radical/far-left/Marxist/extremist")
    print(f"   'radical' family rate: DSA {hard.mean():.4f} vs JD-only {jh.mean():.4f}")
    print("   -> most of the tier gap is the word 'socialist', which for a DSA member is description, not smear.")

    print('\n3. BY OUTLET CLASS -- where the framing actually lives')
    bo = rates(sc, ['outlet_class','tier'])
    print(bo[['outlet_class','tier','n_articles','self_description','loaded_label','threat','loaded_share']].round(3).to_string(index=False))
    for tier in ['DSA','JD-only']:
        R = sc[(sc.tier==tier)&(sc.outlet_class=='right')].loaded_label.astype(float)
        M = sc[(sc.tier==tier)&(sc.outlet_class=='mainstream')].loaded_label.astype(float)
        _, pv = stats.ttest_ind(R, M, equal_var=False)
        print(f'   {tier:8s} right {R.mean():.3f} (n={len(R)}) vs mainstream {M.mean():.3f} (n={len(M)})  p={pv:.2g}')
    L = sc[sc.loaded_label]
    print(f"\n   right-leaning outlets are {(sc.outlet_class=='right').mean()*100:.1f}% of articles "
          f"but {(L.outlet_class=='right').mean()*100:.1f}% of loaded headlines; mainstream contribute "
          f"{(L.outlet_class=='mainstream').mean()*100:.1f}%.")
    bo.to_csv(TABLES_DIR / 'insurgent_framing_by_outlet.csv', index=False)

    print('\n4. BY CYCLE (DSA challengers)')
    bc = rates(sc[sc.tier=='DSA'], 'cycle')
    print(bc[['cycle','n_articles','loaded_label','threat','israel_gaza','loaded_share']].round(3).to_string(index=False))
    bc.to_csv(TABLES_DIR / 'insurgent_framing_by_cycle.csv', index=False)

    print('\n5. STANCE CAVEAT -- the lexicon counts mentions, not stance:')
    for x in sc[sc.loaded_label].title.dropna().head(6):
        print('   ', str(x)[:105])
    print('   Headlines like "Fox News Keeps Accidentally Making Really Good Arguments for Socialism"')
    print('   match the loaded pattern while being sympathetic. Rates over corpora, never single articles.')

10516 headlines, 17 candidates, 2145 domains

1. BY TIER
   tier  n_articles  self_description  loaded_label  threat  identity  israel_gaza  loaded_share
    DSA        2899             0.144         0.017   0.006     0.014        0.037         0.105
JD-only        7617             0.097         0.002   0.006     0.005        0.008         0.021
   loaded_label   DSA 0.0169  JD-only 0.0021  p=1.7e-09
   threat         DSA 0.0059  JD-only 0.0056  p=0.89
   identity       DSA 0.0138  JD-only 0.0050  p=0.00014
   israel_gaza    DSA 0.0373  JD-only 0.0079  p=1.4e-15

2. WHAT THE LOADED LABEL ACTUALLY IS
   of 49 loaded DSA headlines: 29 are bare 'socialist', 14 are radical/far-left/Marxist/extremist
   'radical' family rate: DSA 0.0048 vs JD-only 0.0012
   -> most of the tier gap is the word 'socialist', which for a DSA member is description, not smear.

3. BY OUTLET CLASS -- where the framing actually lives
outlet_class    tier  n_articles  self_description  loaded_label  threat  loaded_s

## 8. Did insurgent races expand the primary electorate?

Part 10 found that Mamdani's strongest districts both added Democrats and voted at far higher rates — 44.8% of enrolled Democrats against 25.7% where he ran weakest. The obvious question is whether insurgent House candidates did the same.

The registration half cannot be asked. **Twelve of the twenty-one cases are in states that do not register voters by party at all** — Illinois (5), Missouri (3), Texas (2), Michigan (1), Ohio (1) — so "did Democratic registration grow" is undefined there, not merely unmeasured, and the nine that remain are a closed-primary subset that would not represent the rest.

Turnout exists everywhere, and it is the analogue of the cleaner half of the NYC result. Democratic House primary votes by district come from the FEC's biennial compilation, each case compared with the same district in the preceding cycle.

**The comparison must be made within cycle.** National primary turnout swings enormously between cycles — the median district ran 1.24× in 2018, 1.53× in 2020 and 0.62× in 2022 — and the case set is concentrated in the two high-growth cycles. Pooling across cycles therefore manufactures a positive result out of composition alone; the cell below shows both so the artifact is visible.

In [9]:
TP = RAW_DIR / 'fec_results' / 'dem_house_primary_turnout.csv'
if not TP.exists():
    print('run src/data_collection/fec_primary_turnout_pull.py first')
else:
    tp = pd.read_csv(TP)
    cc = d[d.type == 'challenger vs incumbent'].copy() if 'type' in d.columns else d.copy()
    cc = cc[cc.contested] if 'contested' in cc.columns else cc
    cc['prior'] = cc.cycle - 2
    j = cc.merge(tp.rename(columns={'cycle': 'cy', 'dem_primary_votes': 'dem_now'})[['cy', 'state', 'district', 'dem_now']],
                 left_on=['cycle', 'state', 'district'], right_on=['cy', 'state', 'district'], how='left')
    j = j.merge(tp.rename(columns={'cycle': 'pc', 'dem_primary_votes': 'dem_prior'})[['pc', 'state', 'district', 'dem_prior']],
                left_on=['prior', 'state', 'district'], right_on=['pc', 'state', 'district'], how='left')
    j['turnout_growth'] = j.dem_now / j.dem_prior
    j['crosses_redistricting'] = j.cycle == 2022
    have = j[j.turnout_growth.notna()].copy()
    print(f'{len(have)} of {len(j)} challenger cases have both cycles.')
    print('  5 are 2026 cases (no FEC compilation published for 2024 or 2026).')
    miss = j[j.turnout_growth.isna() & (j.cycle < 2026)]
    if len(miss):
        print(f'  {len(miss)} more have an UNCONTESTED prior primary, so growth is undefined:')
        for r in miss.itertuples():
            print(f'     {r.cycle} {r.state}-{r.district} {r.candidate}: the incumbent faced no primary the cycle before')
        print('  That is substantive, not missing data -- the challenge created a contest where none existed.')

    print('\nPER CASE:')
    cols = ['cycle', 'state', 'district', 'candidate', 'tier', 'won', 'dem_prior', 'dem_now', 'turnout_growth', 'crosses_redistricting']
    print(have[cols].sort_values('turnout_growth', ascending=False).to_string(index=False))
    have[cols].to_csv(TABLES_DIR / 'insurgent_primary_turnout.csv', index=False)

    # national benchmark, same district pairs
    nat = []
    for cy in [2018, 2020, 2022]:
        a = tp[tp.cycle == cy][['state', 'district', 'dem_primary_votes']].rename(columns={'dem_primary_votes': 'now'})
        b = tp[tp.cycle == cy - 2][['state', 'district', 'dem_primary_votes']].rename(columns={'dem_primary_votes': 'prior'})
        g = a.merge(b, on=['state', 'district']); g['growth'] = g.now / g.prior; g['cycle'] = cy
        nat.append(g)
    nat = pd.concat(nat)
    print('\nWITHIN CYCLE -- insurgent cases against every district with the same cycle pair:')
    rows = []
    for cy in [2018, 2020, 2022]:
        n = nat[nat.cycle == cy].growth.replace([np.inf, -np.inf], np.nan).dropna()
        h = have[have.cycle == cy].turnout_growth.dropna()
        if len(h) == 0: continue
        u = stats.mannwhitneyu(h, n, alternative='greater') if len(h) > 1 else None
        rows.append({'cycle': cy, 'national_median': n.median(), 'national_n': len(n),
                     'insurgent_median': h.median(), 'insurgent_n': len(h),
                     'p_greater': u.pvalue if u is not None else np.nan})
        print(f"  {cy}: national median {n.median():.2f} (n={len(n)})   insurgent median {h.median():.2f} (n={len(h)})"
              + (f"   p={u.pvalue:.3f}" if u is not None else ""))
    pd.DataFrame(rows).to_csv(TABLES_DIR / 'insurgent_turnout_vs_national.csv', index=False)
    print(f"\n  POOLED (misleading): national {nat.growth.replace([np.inf,-np.inf],np.nan).median():.2f}  insurgent {have.turnout_growth.median():.2f}")
    print("  The pooled gap is composition: the case set overweights 2018 and 2020, when every district's")
    print("  primary turnout rose, and underweights 2022, when it collapsed. Within cycle the gap vanishes.")

    print('\nBY TIER AND OUTCOME (small n; descriptive only):')
    for k, lab in [('tier', 'tier'), ('won', 'outcome')]:
        for v, s in have.groupby(k):
            print(f'  {lab} {str(v):8s} n={len(s):2d}  median growth {s.turnout_growth.median():.2f}')

14 of 21 challenger cases have both cycles.
  5 are 2026 cases (no FEC compilation published for 2024 or 2026).
  2 more have an UNCONTESTED prior primary, so growth is undefined:
     2018 FL-7 Chardo Richardson: the incumbent faced no primary the cycle before
     2018 NY-14 Alexandria Ocasio-Cortez: the incumbent faced no primary the cycle before
  That is substantive, not missing data -- the challenge created a contest where none existed.

PER CASE:
 cycle state  district        candidate    tier  won  dem_prior  dem_now  turnout_growth  crosses_redistricting
  2020    NY        16    Jamaal Bowman     DSA  1.0    30078.0  89138.0        2.963561                  False
  2018    MA         7  Ayanna Pressley JD-only  1.0    38213.0 102648.0        2.686206                  False
  2018    CO         1        Saira Rao JD-only  0.0    64695.0 133500.0        2.063529                  False
  2020    MA         1       Alex Morse JD-only  0.0    70354.0 143393.0        2.038164      

### 8b. The 2026 cases, from state sources

The FEC compilation does not exist for 2024 or 2026, so the five 2026 cases need both figures from elsewhere and cannot use the national benchmark. Democratic primary totals for every district in the five relevant states come from the state House-election articles' results tables (`data/raw/fec_results/dem_primary_2024_2026_states.csv`), which lets each insurgent district be compared with **the other districts in its own state and cycle pair** — a benchmark that absorbs the state's turnout environment in place of the national one.

Two of the five have no 2024 baseline, and the reason is the same one that removed Ocasio-Cortez from the earlier table: **the incumbent's 2024 primary was canceled outright.** Espaillat in NY-13 and Foushee in NC-4 both advanced without appearing on a ballot. With AOC's NY-14 in 2018 and Richardson's FL-7 in 2018, that makes four of the twenty-one contested challenges — nearly a fifth — aimed at seats where no primary had been held the cycle before.

In [10]:
SF = RAW_DIR / 'fec_results' / 'dem_primary_2024_2026_states.csv'
if not SF.exists():
    print('run the 2026 state-source extraction first')
else:
    st = pd.read_csv(SF)
    w = st.pivot_table(index=['state', 'district'], columns='cycle', values='dem_primary_votes')
    w = w.dropna(); w['growth'] = w[2026] / w[2024]
    CASES_26 = [('CO', 1, 'Melat Kiros', 'DSA', 1), ('MI', 13, 'Donavan McKinney', 'DSA', 1),
                ('MO', 1, 'Cori Bush', 'DSA', 0), ('NY', 13, 'Darializa Avila Chevalier', 'DSA', 1),
                ('NC', 4, 'Nida Allam', 'JD-only', 0)]
    print('2026 CASES AGAINST THEIR OWN STATE:')
    rows = []
    for ab, dist, name, tier, won in CASES_26:
        peers = w.loc[ab].drop(index=dist, errors='ignore') if ab in w.index.get_level_values(0) else None
        if (ab, dist) not in w.index:
            print(f'  {ab}-{dist:<2d} {name:26s} no 2024 baseline (primary canceled) -- growth undefined')
            rows.append({'state': ab, 'district': dist, 'candidate': name, 'tier': tier, 'won': won,
                         'growth': np.nan, 'state_median': peers.growth.median() if peers is not None and len(peers) else np.nan,
                         'note': 'prior primary canceled'})
            continue
        g = w.loc[(ab, dist), 'growth']
        med = peers.growth.median() if peers is not None and len(peers) else np.nan
        rows.append({'state': ab, 'district': dist, 'candidate': name, 'tier': tier, 'won': won,
                     'growth': g, 'state_median': med, 'note': ''})
        flag = 'ABOVE' if g > med else 'below'
        print(f'  {ab}-{dist:<2d} {name:26s} growth {g:.2f}   state median {med:.2f} (n={len(peers)})   {flag}')
    r26 = pd.DataFrame(rows)
    r26['vs_state'] = r26.growth - r26.state_median
    r26.to_csv(TABLES_DIR / 'insurgent_turnout_2026.csv', index=False)
    print('\n  Two of the three measurable 2026 cases fall BELOW their state median; one is above.')
    print('  Cori Bush\'s MO-1 comeback had the second-lowest growth of any Missouri district.')

    print('\nSEATS WITH NO PRIOR CONTEST (the pattern the ratio cannot represent):')
    NOPRIOR = [('2018 NY-14', 'Ocasio-Cortez', 'won'), ('2018 FL-7', 'Richardson', 'lost'),
               ('2026 NY-13', 'Avila Chevalier', 'won'), ('2026 NC-4', 'Allam', 'lost')]
    for case, who, res in NOPRIOR:
        print(f'   {case:11s} {who:16s} {res}')
    print('   4 of 21 contested challenges, and 2 of the 8 wins, came in seats where the incumbent')
    print('   had faced no primary at all the cycle before. The challenge created the contest.')

2026 CASES AGAINST THEIR OWN STATE:
  CO-1  Melat Kiros                growth 2.01   state median 1.69 (n=7)   ABOVE
  MI-13 Donavan McKinney           growth 1.38   state median 1.58 (n=5)   below
  MO-1  Cori Bush                  growth 1.14   state median 1.59 (n=7)   below
  NY-13 Darializa Avila Chevalier  no 2024 baseline (primary canceled) -- growth undefined
  NC-4  Nida Allam                 no 2024 baseline (primary canceled) -- growth undefined

  Two of the three measurable 2026 cases fall BELOW their state median; one is above.
  Cori Bush's MO-1 comeback had the second-lowest growth of any Missouri district.

SEATS WITH NO PRIOR CONTEST (the pattern the ratio cannot represent):
   2018 NY-14  Ocasio-Cortez    won
   2018 FL-7   Richardson       lost
   2026 NY-13  Avila Chevalier  won
   2026 NC-4   Allam            lost
   4 of 21 contested challenges, and 2 of the 8 wins, came in seats where the incumbent
   had faced no primary at all the cycle before. The challenge c

## 9. What the outside money says: super PAC ads against progressive Democrats

Section 3 shows *who* spends against insurgents; this section asks what the money says. Two data sets:

1. **Spending** — every itemized independent expenditure by AIPAC's United Democracy Project, DMFI PAC, the crypto industry's Fairshake and its Democratic-side affiliate Protect Progress, 2022–2026 (`superpac_ie_pull.py`; totals reconcile to the FEC's committee totals for every cycle).
2. **Ads** — a hand-coded set of every ad described in a source in the 32 Democratic primaries where one of the four spent $1M or more (`data/coded/superpac_ads_coded.csv`, codebook `src/analysis/attack_modes.py`, written before any ad was read). Sources are the committees' own ad-citation pages (UDP publishes one per ad; DMFI a few) and reporting that quotes or describes the ad. Each ad carries every mode that applies and a lead mode; five modes emerged in coding and are defined with the ads that prompted them.

Four hypotheses were fixed in advance: (1) the committees rarely name their own issue; (2) that is not specific to pro-Israel money; (3) it predates the 2024–26 shift in opinion on Gaza; (4) terrorism and identity-coded attacks fall on Muslim and Arab targets more than others. Off-season ads, Republican primaries and races under $1M are kept as contrast cases, outside the counts.

In [11]:
import sys
from pathlib import Path
import numpy as np, pandas as pd


def superpac_money_and_ads():
    """Kept in a function so its frames (d, c, M) do not overwrite the notebook's case tables."""
    ROOT = Path('.') if Path('data').exists() else Path('..')
    sys.path.insert(0, str(ROOT / 'src'))
    from analysis.attack_modes import load, MODES, EMERGENT, SPENDER_ISSUE
    RAW = ROOT / 'data' / 'raw' / 'superpac_ads'; CODED = ROOT / 'data' / 'coded'; TABLES = ROOT / 'outputs' / 'tables'
    pd.set_option('display.width', 220)

    # ---------------- A. how the money is spent ----------------
    ie = pd.read_csv(RAW / 'superpac_ie_itemized.csv'); ie = ie[ie.cycle >= 2022]
    ie['stage'] = ie.election_type.fillna('').str[0].map({'P': 'primary', 'G': 'general', 'R': 'primary', 'S': 'primary'}).fillna('other')
    ie['issue'] = ie.spender.map(SPENDER_ISSUE)
    ie['bucket'] = np.select([(ie.party == 'DEM') & (ie.stage == 'primary') & (ie.support_oppose == 'O'),
                              (ie.party == 'DEM') & (ie.stage == 'primary') & (ie.support_oppose == 'S'),
                              (ie.party == 'DEM') & (ie.stage != 'primary'),
                              ie.party == 'REP'],
                             ['Dem primary: against', 'Dem primary: for', 'Dem general', 'Republicans'], 'other')
    A = ie.pivot_table(index='spender', columns='bucket', values='amount', aggfunc='sum', fill_value=0) / 1e6
    A['total'] = A.sum(axis=1)
    M = ie.pivot_table(index='spender', columns='medium', values='amount', aggfunc='sum', fill_value=0)
    M = M.div(M.sum(axis=1), axis=0)
    print('A. SPENDING 2022-2026, $M'); print(A.round(1).to_string())
    print('\n   medium share'); print(M.round(3).to_string())
    A.round(2).to_csv(TABLES / 'superpac_spending_by_bucket.csv'); M.round(4).to_csv(TABLES / 'superpac_spending_by_medium.csv')

    # ---------------- B. what the ads say ----------------
    d, modes = load(CODED / 'superpac_ads_coded.csv')
    u = d[d.in_universe & d.duplicate_of.isna()].copy()
    u['attack'] = ~u.lead.eq('positive_other')
    print(f'\nB. ADS: {len(u)} in universe, {u.race.nunique()} races, {u.attack.sum()} attack, {(~u.attack).sum()} promote')
    carry = u.groupby('issue')[modes].mean().T
    carry['all'] = u[modes].mean()
    carry.loc['N ads'] = u.groupby('issue').size().tolist() + [len(u)]
    print(carry.round(2).to_string())
    lead = pd.crosstab(u.lead, u.issue, margins=True)
    print('\n   lead mode (counts)'); print(lead.to_string())
    carry.round(3).to_csv(TABLES / 'superpac_ad_modes.csv'); lead.to_csv(TABLES / 'superpac_ad_lead_mode.csv')

    # hypothesis 1-2: own issue
    att = u[u.attack]
    print('\n   H1/H2 own issue named, all ads:', u.groupby('issue').own_issue.agg(['sum', 'size']).to_dict('index'))
    print('   own issue named, attack ads:', att.groupby('issue').own_issue.agg(['sum', 'size']).to_dict('index'))
    # hypothesis 3: 2022 vs later, Israel committees
    isr = u[u.issue == 'israel']
    print('   H3 Israel committees, own issue by cycle:', isr.groupby('cycle').own_issue.agg(['sum', 'size']).to_dict('index'))
    # contrast cases
    c = d[~d.in_universe]
    print('   contrast cases (off-season petitions, Republican primaries, sub-$1M):')
    print(c[['race', 'title', 'own_issue']].to_string(index=False))
    # disputed
    print(f"\n   attack ads whose central claim a named outlet disputed: {att.disputed_by.notna().sum()} of {len(att)}")
    print('   by committee type:', att.groupby('issue').disputed_by.apply(lambda s: f'{s.notna().sum()}/{len(s)}').to_dict())

    # hypothesis 4: identity
    MUSLIM_ARAB = {'El-Sayed', 'Allam', 'Wahab', 'Campa-Najjar'}   # Muslim or of Arab descent, per public biography
    att = att.assign(muslim_arab=att.target.isin(MUSLIM_ARAB))
    H4 = att.groupby('muslim_arab')[['terror_association', 'identity_coded', 'guilt_by_association']].agg(['sum', 'size'])
    print('\n   H4 attack ads by target (Muslim or Arab):'); print(H4.to_string())

    # ---------------- coverage: how much of the money the coded ads speak for ----------------
    dp = ie[(ie.party == 'DEM') & (ie.stage == 'primary')].copy()
    dp['race'] = dp.cycle.astype(str) + ' ' + dp.state + '-' + np.where(dp.office == 'H', dp.district.fillna(0).astype(int).astype(str), 'SEN')
    covered = set(u.race)
    cov = dp.assign(doc=dp.race.isin(covered)).groupby(['issue', 'doc']).amount.sum().unstack(fill_value=0)
    cov['share_documented'] = cov[True] / cov.sum(axis=1)
    print('\nCoverage: share of Dem-primary dollars in races with at least one documented ad'); print(cov.round(3).to_string())



superpac_money_and_ads()


A. SPENDING 2022-2026, $M
bucket            Dem general  Dem primary: against  Dem primary: for  Republicans  other  total
spender                                                                                         
DMFI                      0.7                   5.7               5.9          1.3    0.4   14.1
Fairshake                11.1                  26.8               0.6         13.1    2.3   54.0
Protect Progress         20.9                   2.8              38.5          0.0    0.0   62.2
UDP                       4.1                  57.4              47.0          5.7    3.3  117.4

   medium share
medium            digital  field   mail  other  production  radio     tv
spender                                                                 
DMFI                0.270  0.011  0.143  0.187       0.007  0.029  0.353
Fairshake           0.000  0.000  0.007  0.003       0.011  0.004  0.975
Protect Progress    0.001  0.000  0.097  0.000       0.010  0.000  0.891
UDP       

## 10. Is the left gaining? 2026 against 2018–2024

A test of the claim that momentum in 2025–26 is running toward the democratic-socialist left and away from moderates, with what would count for and against it written down before computing:

- **M1** — DSA-tier candidates win more often in 2026 than 2018–24 in the same race type.
- **M2** — in contested races, 2026 DSA-tier candidates run further above the money model's prediction than earlier ones (model fitted on 2018–24 only).
- **M3** — being outspent costs less in 2026.
- **M4** — incumbents lose to their left more often in 2026, not only more often.
- **Targets** — the candidates the pro-Israel and crypto super PACs spent against in 2026 beat the money more than their 2022–24 targets (`superpac_targets_pull.py`: 24 races with an identifiable target, head-to-head share against the main rival, money on both sides counted as in Part 11).
- **Age** — ideology and generation move together in 2026; the challenger-incumbent age gap (`candidate_ages_pull.py`, Wikidata birth dates, 18 of the 21 challenger cases) is added to the challenger model to see whether it absorbs the DSA edge.

In [12]:
import scipy.stats as st
import numpy as np, pandas as pd, statsmodels.api as sm, scipy.stats as st
from pathlib import Path
import sys


def momentum_tests(c, m):
    """Kept in a function so its frames do not overwrite the notebook's case tables."""
    # ---------------- M1-M3: the Justice Democrats cases, 2026 against 2018-24 ----------------
    cc = c.copy(); cc['period'] = np.where(cc.cycle >= 2026, '2026', '2018-24')
    print('M1. Win rate by tier, race type and period')
    print(cc.groupby(['tier', 'type', 'period']).agg(N=('won', 'size'), wins=('won', 'sum'), rate=('won', 'mean')).round(2).to_string())
    print('\n    challengers who beat a sitting Democrat, by cycle:', cc[(cc.type == 'challenger vs incumbent') & (cc.won == 1)].groupby('cycle').candidate.apply(list).to_dict())
    mm = m.copy(); mm['period'] = np.where(mm.cycle >= 2026, '2026', '2018-24')
    XM = ['log_ratio', 'log_ie_oppose', 'vs_inc', 'jd_inc', 'dsa']
    b0 = mm[mm.cycle < 2026]
    r0 = sm.OLS(b0.pct, sm.add_constant(b0[XM])).fit(cov_type='HC1')
    mm['resid'] = mm.pct - r0.predict(sm.add_constant(mm[XM]))
    kk = mm[mm.type != 'JD incumbent']
    print('\nM2. Contested races (challengers and open seats): points above the money model fitted on 2018-24')
    print(kk.groupby(['tier', 'period']).resid.agg(['mean', 'count']).round(1))
    for tier in ['DSA', 'JD-only']:
        a_, b_ = kk[(kk.tier == tier) & (kk.period == '2026')].resid, kk[(kk.tier == tier) & (kk.period != '2026')].resid
        print(f'    {tier}: 2026 {a_.mean():+.1f} (n={len(a_)}) vs 2018-24 {b_.mean():+.1f} (n={len(b_)}), p={st.ttest_ind(a_, b_, equal_var=False).pvalue:.2f}')
    mm['y26'] = (mm.cycle >= 2026).astype(float); mm['lr_x26'] = mm.log_ratio * mm.y26
    r3 = sm.OLS(mm.pct, sm.add_constant(mm[XM + ['y26', 'lr_x26']])).fit(cov_type='HC1')
    print(f'\nM3. Cost of being outspent (per log unit of money ratio): 2018-24 {r3.params.log_ratio:.2f} (SE {r3.bse.log_ratio:.2f}); '
          f'2026 difference {r3.params.lr_x26:+.2f} (SE {r3.bse.lr_x26:.2f}, p={r3.pvalues.lr_x26:.2f})')
    kk.groupby(['tier', 'period']).resid.agg(['mean', 'count']).round(2).to_csv(TABLES_DIR / 'momentum_jd_residuals.csv')

    ROOT = Path('..')
    RAW = ROOT / 'data' / 'raw'; TABLES = ROOT / 'outputs' / 'tables'
    pd.set_option('display.width', 220)

    # ---------------- A. Do the super PACs' 2026 targets beat the money more than earlier targets? ----------------
    t = pd.read_csv(RAW / 'superpac_ads' / 'superpac_targets.csv')
    # incumbency of each side (member-vs-member races have both)
    INC_T = {'Andy Levin', 'Jamaal Bowman', 'Al Green'}
    INC_T24 = {('2024', 'Cori Bush')}
    t['target_inc'] = t.target.isin(INC_T) | [(str(c), n) in INC_T24 for c, n in zip(t.cycle, t.target)]
    INC_R = {'Haley Stevens', 'Henry Cuellar', 'Wesley Bell', 'Christian Menefee', 'Shri Thanedar', 'Ritchie Torres'}
    t['rival_inc'] = t.rival.isin(INC_R) & ~((t.cycle == 2022) & (t.rival == 'Shri Thanedar')) & ~((t.cycle == 2024) & (t.rival == 'Wesley Bell'))
    t['log_ratio'] = np.log(t.money_ratio.clip(0.05, 50))
    t['y2026'] = (t.cycle == 2026).astype(float)
    t['inc_diff'] = t.target_inc.astype(float) - t.rival_inc.astype(float)
    print('A. SUPER PAC TARGETS: head-to-head share on money, incumbency and 2026')
    X = sm.add_constant(t[['log_ratio', 'inc_diff', 'y2026']])
    ra = sm.OLS(t.h2h * 100, X).fit(cov_type='HC1')
    print(pd.DataFrame({'coef': ra.params, 'se': ra.bse, 'p': ra.pvalues}).round(2))
    base = t[t.cycle < 2026]
    rb = sm.OLS(base.h2h * 100, sm.add_constant(base[['log_ratio', 'inc_diff']])).fit(cov_type='HC1')
    t['pred'] = rb.predict(sm.add_constant(t[['log_ratio', 'inc_diff']])); t['resid'] = t.h2h * 100 - t.pred
    S = t.groupby('cycle').agg(N=('won', 'size'), won=('won', 'sum'), median_ratio=('money_ratio', 'median'),
                                mean_h2h=('h2h', 'mean'), mean_resid=('resid', 'mean')).round(2)
    print('\n   by cycle (residual vs a model fitted on 2022-24 only)'); print(S)
    a, b = t[t.cycle == 2026].resid, t[t.cycle < 2026].resid
    print(f'   2026 vs 2022-24 residual: {a.mean():+.1f} vs {b.mean():+.1f}, p={st.ttest_ind(a, b, equal_var=False).pvalue:.2f}')
    L = t[t.left == True]
    print('   left targets only:', L.groupby(L.cycle == 2026).agg(N=('won', 'size'), won=('won', 'sum'), resid=('resid', 'mean')).round(1).to_dict('index'))
    cols = ['race', 'target', 'rival', 'left', 'won', 'h2h', 'money_ratio', 'pred', 'resid']
    print(t[cols].round(2).to_string(index=False))
    t[cols + ['target_inc', 'rival_inc']].round(3).to_csv(TABLES / 'superpac_targets_scored.csv', index=False)
    S.to_csv(TABLES / 'superpac_targets_by_cycle.csv')

    # ---------------- B. Is the DSA edge age? ----------------
    ages = pd.read_csv(RAW / 'insurgent' / 'candidate_ages.csv')
    dob = ages.dropna(subset=['dob']).drop_duplicates('name').set_index('name').dob.map(pd.Timestamp)
    sys.path.insert(0, str(ROOT / 'src' / 'data_collection'))
    from candidate_ages_pull import ALIAS, DEFEATS_2026
    k = m[m.type == 'challenger vs incumbent'].copy()          # `m`: the section 4 model frame
    k['inc_name'] = k.opp_name.map(lambda n: ALIAS.get(n, n.title()))
    edate = pd.to_datetime(k.cycle.astype(str) + '-06-30')
    k['chal_age'] = (edate - k.candidate.map(dob)).dt.days / 365.25
    k['inc_age'] = (edate - k.inc_name.map(dob)).dt.days / 365.25
    k['age_gap'] = k.inc_age - k.chal_age
    k = k.dropna(subset=['age_gap'])
    print('');print(f'B. AGE: {len(k)} of 21 challenger-vs-incumbent cases with both ages')
    print(k.groupby('tier')[['chal_age', 'inc_age', 'age_gap', 'pct', 'won']].mean().round(1))
    for lab, cols_ in [('money + DSA', ['log_ratio', 'dsa']), ('money + age gap', ['log_ratio', 'age_gap']),
                       ('money + DSA + age gap', ['log_ratio', 'dsa', 'age_gap'])]:
        r = sm.OLS(k.pct, sm.add_constant(k[cols_])).fit(cov_type='HC1')
        print(f'   {lab:24s} ' + '  '.join(f'{c} {r.params[c]:+.2f} ({r.bse[c]:.2f})' for c in cols_) + f'   R2 {r.rsquared:.2f}')
    print(f"   correlation of DSA tier with age gap: {k[['dsa', 'age_gap']].corr().iloc[0, 1]:.2f}")
    k[['cycle', 'candidate', 'inc_name', 'tier', 'won', 'pct', 'chal_age', 'inc_age', 'age_gap']].round(1).to_csv(TABLES / 'insurgent_age_gap.csv', index=False)
    D = pd.DataFrame(DEFEATS_2026, columns=['incumbent', 'winner', 'kind'])
    D['inc_age'] = ((pd.Timestamp('2026-06-30') - D.incumbent.map(dob)).dt.days / 365.25).round(0)
    D['winner_age'] = ((pd.Timestamp('2026-06-30') - D.winner.map(dob)).dt.days / 365.25).round(0)
    print('');print('   2026 House Democratic incumbents defeated in primaries'); print(D.to_string(index=False))
    D.to_csv(TABLES / 'defeats_2026_ages.csv', index=False)



# the case table from section 1 (`c` is reused as a loop variable in section 7)
momentum_tests(d[d.contested].copy(), m)


M1. Win rate by tier, race type and period
                                          N  wins  rate
tier    type                    period                 
DSA     JD incumbent            2018-24  12  10.0  0.83
                                2026      5   5.0  1.00
        challenger vs incumbent 2018-24   4   3.0  0.75
                                2026      4   3.0  0.75
        open / other            2018-24   5   3.0  0.60
                                2026      2   2.0  1.00
JD-only JD incumbent            2018-24   9   8.0  0.89
                                2026      2   2.0  1.00
        challenger vs incumbent 2018-24  12   2.0  0.17
                                2026      1   0.0  0.00
        open / other            2018-24  44  11.0  0.25
                                2026      6   3.0  0.50

    challengers who beat a sitting Democrat, by cycle: {2018: ['Ayanna Pressley', 'Alexandria Ocasio-Cortez'], 2020: ['Marie Newman', 'Cori Bush', 'Jamaal Bowman'], 2026: [

## 11. Generational realignment: every contested Democratic incumbent primary, 2024 and 2026

Section 10 found that age does not separate democratic-socialist from conventional-progressive challengers, but three of 2026's DSA upsets were won by challengers some 40 years younger than the incumbent, and John Larson, 78, lost to a younger moderate. This section asks the broader question on every contested Democratic House incumbent primary in 2024 and 2026 (`incumbent_primaries_pull.py`: Wikipedia results via the MediaWiki API; ACS 2024 district age; incumbent birth dates from Wikidata by article link). 2026 races in states that redrew mid-decade are excluded, as are California and Washington's all-party primaries.

Written before the pull: **H1** in 2026 the incumbent's share falls with the district's 18–34 share, net of incumbent age; **H2** that gradient is steeper than in 2024; **H3** incumbent age lowers the share net of district youth.

In [13]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def generations_test():
    """Kept in a function so its frames do not overwrite the notebook's case tables."""
    g = pd.read_csv(RAW_DIR / 'insurgent' / 'incumbent_primaries.csv')
    g = g.dropna(subset=['young_share', 'inc_age']).copy()
    g['share'] = 100 * g.inc_share; g['young'] = 100 * g.young_share
    g['log_n'] = np.log(g.n_candidates); g['y26'] = (g.year == 2026).astype(float)
    g['young_x26'] = g.young * g.y26; g['age_x26'] = g.inc_age * g.y26
    print('Contested Democratic incumbent primaries with district and incumbent age')
    print(g.groupby('year').agg(races=('lost', 'size'), lost=('lost', 'sum'), mean_share=('share', 'mean'),
                                young=('young', 'mean'), inc_age=('inc_age', 'mean')).round(1))
    rows = []
    for yr in (2024, 2026):
        s = g[g.year == yr]
        m = smf.ols('share ~ young + inc_age + log_n', data=s).fit(cov_type='HC1')
        rows.append(dict(model=f'{yr}', N=int(m.nobs), young=m.params.young, young_se=m.bse.young, young_p=m.pvalues.young,
                         inc_age=m.params.inc_age, inc_age_se=m.bse.inc_age, inc_age_p=m.pvalues.inc_age))
    m = smf.ols('share ~ young + inc_age + log_n + y26 + young_x26 + age_x26', data=g).fit(cov_type='HC1')
    rows.append(dict(model='pooled: 2026 difference', N=int(m.nobs), young=m.params.young_x26, young_se=m.bse.young_x26,
                     young_p=m.pvalues.young_x26, inc_age=m.params.age_x26, inc_age_se=m.bse.age_x26, inc_age_p=m.pvalues.age_x26))
    R = pd.DataFrame(rows)
    print('\nIncumbent primary share (points) on the district 18-34 share (per point) and incumbent age (per year),')
    print('net of the number of candidates; HC1')
    print(R.round(3).to_string(index=False))
    R.to_csv(TABLES_DIR / 'generations_models.csv', index=False)
    # the same districts in both years
    w = g.pivot_table(index=['state', 'district'], columns='year', values=['share', 'young', 'inc_age'])
    w = w.dropna()
    w.columns = [f'{a}_{b}' for a, b in w.columns]
    w['d_share'] = w.share_2026 - w.share_2024
    if len(w) >= 8:
        m = smf.ols('d_share ~ young_2024 + inc_age_2026', data=w).fit(cov_type='HC1')
        print(f'\nSame districts contested both years (N={len(w)}): change in incumbent share on young share '
              f'{m.params.young_2024:+.3f} ({m.bse.young_2024:.3f}), incumbent age {m.params.inc_age_2026:+.3f} ({m.bse.inc_age_2026:.3f})')
    g['young_q'] = pd.qcut(g.young, 4, labels=['oldest', '2', '3', 'youngest'])
    print('\nIncumbent share and losses by district youth quartile')
    print(g.groupby(['year', 'young_q'], observed=True).agg(N=('lost', 'size'), lost=('lost', 'sum'),
                                                            share=('share', 'mean')).round(1).unstack(0))
    g[['year', 'state', 'district', 'incumbent', 'share', 'lost', 'young', 'inc_age', 'n_candidates']].round(2).to_csv(
        TABLES_DIR / 'generations_incumbents.csv', index=False)


generations_test()


Contested Democratic incumbent primaries with district and incumbent age
      races  lost  mean_share  young  inc_age
year                                         
2024     39     2        78.1   31.8     60.7
2026     48     5        71.9   30.2     60.0

Incumbent primary share (points) on the district 18-34 share (per point) and incumbent age (per year),
net of the number of candidates; HC1
                  model  N  young  young_se  young_p  inc_age  inc_age_se  inc_age_p
                   2024 39 -0.312     0.493    0.527    0.151       0.169      0.372
                   2026 48 -1.061     0.613    0.083   -0.164       0.178      0.358
pooled: 2026 difference 87 -0.766     0.789    0.332   -0.239       0.225      0.289

Same districts contested both years (N=14): change in incumbent share on young share -0.471 (1.153), incumbent age -0.601 (0.343)

Incumbent share and losses by district youth quartile
            N      lost      share      
year     2024 2026 2024 2026  2024 

## Limitations

- **Universe = Justice Democrats endorsements.** Insurgent-left candidates JD did not endorse (e.g., Brad Lander 2026, Shahid Buttar 2020) are excluded; JD's 2018 slate included many open-seat and general-election-only candidates, which is why the analysis conditions on contested primaries and separates the challenger-vs-incumbent cases.
- **Opponent identification is mechanical:** the Democratic incumbent, else the best-funded other Democrat by cycle receipts. In multi-candidate open seats the "opponent" is one of several.
- **Pre-primary receipts are year-to-date at the last report before the primary plus the prior year-end;** where the FEC has no 12P report the prior quarterly is used. Independent expenditures are cycle totals, which for incumbents include general-election spending; for primary-only losers they are effectively primary totals.
- **2026 outcomes are as recorded on Wikipedia at pull time**; 2026 FEC totals are partial (cycle still open).
- **The DSA tier is hand-coded** from Ballotpedia's DSA endorsement tables, DSA statements, and candidates' articles; chapter endorsements count. Fourteen of 133 cases.
- **Small N.** Roughly 25 challenger-vs-incumbent cases across five cycles; the models are descriptive.